# PLM Optimized Storage Controller — Method-level tests

Tests each method of `PeakLoadManagementOptimizedStorageController` in isolation
using small synthetic data. No full OpenMDAO model needed.

In [ ]:
import sys
from pathlib import Path

# Add the repo root to sys.path so the h2integrate package is importable
repo_root = Path("../../").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import pyomo.environ as pyomo
import matplotlib.pyplot as plt

from h2integrate.control.control_strategies.storage.plm_optimized_storage_controller import (
    PeakLoadManagementOptimizedControllerConfig,
    PeakLoadManagementOptimizedStorageController,
)

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

## Helper: build a lightweight controller

Bypasses OpenMDAO `setup()` so each method can be tested directly.

In [ ]:
N = 48  # 2 days of hourly timesteps
DT_SECONDS = 3600  # 1-hour resolution

# Synthetic LMP signal: flat with spikes at hours 15 and 37 (peak window 14-18)
lmp = np.full(N, 20.0)
lmp[15] = 80.0  # day-1 peak inside peak window
lmp[14] = 60.0
lmp[16] = 55.0
lmp[37] = 75.0  # day-2 peak inside peak window (hour 13 of day 2)
lmp[36] = 50.0
lmp[38] = 45.0

config = PeakLoadManagementOptimizedControllerConfig(
    max_capacity=1200.0,
    max_charge_rate=300.0,
    max_soc_fraction=0.90,
    min_soc_fraction=0.10,
    init_soc_fraction=0.90,
    charge_efficiency=0.95,
    discharge_efficiency=0.95,
    n_control_window=N,  # full 2-day window as one solve
    commodity="electricity",
    commodity_rate_units="kW",
    tech_name="battery",
    system_commodity_interface_limit=1e9,
    supervisory_signal=lmp.tolist(),
    peak_window={"start": "14:00:00", "end": "18:00:00"},
    performance_incentive={"units": "$/kWh", "val": 14.0},
    n_max_events=10,
    signal_threshold_percentile=95.0,
    event_duration={"units": "h", "val": 2},  # +/- 1h around daily peak
)


def make_controller(cfg=config, n=N, dt=DT_SECONDS):
    """Instantiate the controller without OpenMDAO infrastructure."""
    ctrl = object.__new__(PeakLoadManagementOptimizedStorageController)
    ctrl.config = cfg
    ctrl.dt_seconds = dt
    ctrl.n_timesteps = n
    ctrl.updated_initial_soc = cfg.init_soc_fraction
    ctrl.time_index = pd.date_range("2025-01-01", periods=n, freq=pd.to_timedelta(dt, unit="s"))
    ctrl.in_peak_window = ctrl._compute_peak_window_mask()
    ctrl.month_ids = ctrl._compute_month_ids()
    return ctrl


ctrl = make_controller()
print("Controller created. time_index:", ctrl.time_index[[0, -1]].tolist())

## 1. `_parse_peak_window`

In [ ]:
start, end = ctrl._parse_peak_window()
print(f"peak_window start={start}  end={end}")
assert start.hour == 14 and end.hour == 18

## 2. `_compute_peak_window_mask`

In [ ]:
mask = ctrl.in_peak_window
print("Timesteps inside peak window:", np.where(mask)[0].tolist())
# Expect hours 14-18 on each day: [14,15,16,17,18,38,39,40,41,42]

fig, ax = plt.subplots(figsize=(10, 2))
ax.fill_between(range(N), mask.astype(int), step="post", alpha=0.4, label="peak window")
ax.set_xlabel("Timestep (h)")
ax.set_title("Static peak-window mask")
ax.legend()
plt.tight_layout()
plt.show()

## 3. `_compute_month_ids`

In [ ]:
month_ids = ctrl._compute_month_ids()
print("Unique months in 2-day window:", np.unique(month_ids).tolist())
assert set(month_ids) == {1}  # both days are January

## 4. `_compute_eligible_mask`

Percentile-based signal filter applied to a rolling window.

In [ ]:
signal_window = np.asarray(config.supervisory_signal)

# a) No dispatch_mask — percentile over full window
eligible_full = ctrl._compute_eligible_mask(signal_window)
print("Eligible (full window, p=95%):", np.where(eligible_full)[0].tolist())

# b) dispatch_mask = peak_window only — percentile over in-window signals
eligible_pw = ctrl._compute_eligible_mask(signal_window, dispatch_mask=ctrl.in_peak_window)
print("Eligible (peak-window signals only, p=95%):", np.where(eligible_pw)[0].tolist())

## 5. `_compute_event_window_mask`

Computes the ±`event_duration/2` window centred on each day's LMP peak.

In [ ]:
signal_window = np.asarray(config.supervisory_signal)

event_mask = ctrl._compute_event_window_mask(signal_window, ctrl.in_peak_window, window_start=0)

print("Static peak-window timesteps:", np.where(ctrl.in_peak_window)[0].tolist())
print("Event-window   timesteps    :", np.where(event_mask)[0].tolist())
# Day 1 peak at t=15 → event window = [14, 15, 16] (±1 h)
# Day 2 peak at t=37 → event window = [36, 37, 38] (±1 h)

fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(signal_window, color="steelblue", linewidth=1)
axes[0].set_ylabel("LMP ($/MWh)")
axes[0].set_title("Signal")

axes[1].fill_between(
    range(N), ctrl.in_peak_window.astype(int), step="post", alpha=0.3, label="static peak_window"
)
axes[1].fill_between(
    range(N),
    event_mask.astype(int),
    step="post",
    alpha=0.5,
    color="darkorange",
    label="event window (±1 h)",
)
axes[1].set_xlabel("Timestep (h)")
axes[1].set_title("Masks")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. `_build_dr_model` + solve

Build and solve the MILP for the full 2-day window. Expect discharge at event-window timesteps.

In [ ]:
# Expand system_commodity_interface_limit to match window length
object.__setattr__(
    ctrl.config,
    "system_commodity_interface_limit",
    [1e9] * N,
)

model = ctrl._build_dr_model(
    window_start=0,
    window_len=N,
    init_soc=config.init_soc_fraction,
    remaining_budget={1: config.n_max_events},
)

results = PeakLoadManagementOptimizedStorageController.glpk_solve_call(model)
print("Solver status:", results.solver.status)
print("Termination :", results.solver.termination_condition)

In [ ]:
discharge = np.array([pyomo.value(model.discharge[t]) for t in range(N)])
charge = np.array([pyomo.value(model.charge[t]) for t in range(N)])
soc = np.array([pyomo.value(model.soc[t]) for t in range(N)])

dispatch_ts = np.where(discharge > 0.5)[0].tolist()
charge_ts = np.where(charge > 0.5)[0].tolist()
print("Discharge timesteps:", dispatch_ts)
print("Charge    timesteps:", charge_ts)
print(f"Total discharge events: {int(discharge.sum())}  (limit: {config.n_max_events}/month)")

# --- verify discharge is within event window ---
outside = [t for t in dispatch_ts if not event_mask[t]]
print("\nDischarge outside event window (expect empty):", outside)
assert not outside, f"Unexpected discharge outside event window: {outside}"

In [ ]:
P_max = config.max_charge_rate
net_power = (discharge - charge) * P_max

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

ax = axes[0]
ax.plot(signal_window, color="steelblue", linewidth=1, label="LMP")
ax.fill_between(
    range(N),
    event_mask * signal_window.max(),
    step="post",
    alpha=0.25,
    color="darkorange",
    label="event window",
)
ax.set_ylabel("LMP ($/MWh)")
ax.legend(fontsize=8)

ax = axes[1]
ax.bar(range(N), discharge * P_max, color="tomato", label="discharge (kW)")
ax.bar(range(N), -charge * P_max, color="steelblue", label="charge (kW)")
ax.set_ylabel("Power (kW)")
ax.legend(fontsize=8)

ax = axes[2]
ax.plot(soc * 100, color="green", linewidth=1.2)
ax.axhline(config.max_soc_fraction * 100, color="gray", linestyle=":", linewidth=0.8)
ax.axhline(config.min_soc_fraction * 100, color="gray", linestyle=":", linewidth=0.8)
ax.set_ylabel("SOC (%)")
ax.set_xlabel("Timestep (h)")
ax.set_ylim([0, 105])

plt.tight_layout()
plt.show()

## 7. Sensitivity: event_duration = None (static peak window)

With no `event_duration`, dispatch should cover all timesteps in the static peak window
whose signal exceeds the percentile threshold.

In [ ]:
config_no_dur = PeakLoadManagementOptimizedControllerConfig(
    max_capacity=1200.0,
    max_charge_rate=300.0,
    max_soc_fraction=0.90,
    min_soc_fraction=0.10,
    init_soc_fraction=0.90,
    charge_efficiency=0.95,
    discharge_efficiency=0.95,
    n_control_window=N,
    commodity="electricity",
    commodity_rate_units="kW",
    tech_name="battery",
    system_commodity_interface_limit=1e9,
    supervisory_signal=lmp.tolist(),
    peak_window={"start": "14:00:00", "end": "18:00:00"},
    performance_incentive={"units": "$/kWh", "val": 14.0},
    n_max_events=10,
    signal_threshold_percentile=95.0,
    event_duration=None,
)

ctrl_no_dur = make_controller(config_no_dur)
object.__setattr__(ctrl_no_dur.config, "system_commodity_interface_limit", [1e9] * N)

model_no_dur = ctrl_no_dur._build_dr_model(
    window_start=0,
    window_len=N,
    init_soc=config_no_dur.init_soc_fraction,
    remaining_budget={1: config_no_dur.n_max_events},
)
PeakLoadManagementOptimizedStorageController.glpk_solve_call(model_no_dur)

discharge_no_dur = np.array([pyomo.value(model_no_dur.discharge[t]) for t in range(N)])
print("Discharge timesteps (no event_duration):", np.where(discharge_no_dur > 0.5)[0].tolist())
print("  (should be 1 or 2 timesteps with highest signal inside peak_window)")

## 8. Sub-hourly resolution test (30-min timesteps)

Verifies that `event_duration` and `n_control_window` scale correctly at any `dt`.

In [ ]:
N_30 = 96  # 2 days at 30-min resolution
DT_30 = 1800  # 30 minutes

lmp_30 = np.full(N_30, 20.0)
# Day 1: spike at 30-min timestep 30 = hour 15
lmp_30[30] = 80.0
lmp_30[28] = 60.0
lmp_30[32] = 55.0
# Day 2: spike at 30-min timestep 74 = hour 37 of 30-min index
lmp_30[74] = 75.0
lmp_30[72] = 50.0
lmp_30[76] = 45.0

config_30 = PeakLoadManagementOptimizedControllerConfig(
    max_capacity=1200.0,
    max_charge_rate=300.0,
    max_soc_fraction=0.90,
    min_soc_fraction=0.10,
    init_soc_fraction=0.90,
    charge_efficiency=0.95,
    discharge_efficiency=0.95,
    n_control_window=N_30,
    commodity="electricity",
    commodity_rate_units="kW",
    tech_name="battery",
    system_commodity_interface_limit=1e9,
    supervisory_signal=lmp_30.tolist(),
    peak_window={"start": "14:00:00", "end": "18:00:00"},
    performance_incentive={"units": "$/kWh", "val": 14.0},
    n_max_events=10,
    signal_threshold_percentile=0.0,  # disable for clarity
    event_duration={"units": "h", "val": 2},  # expect ±1h = ±2 timesteps
)

ctrl_30 = make_controller(config_30, n=N_30, dt=DT_30)
object.__setattr__(ctrl_30.config, "system_commodity_interface_limit", [1e9] * N_30)

event_mask_30 = ctrl_30._compute_event_window_mask(
    np.asarray(lmp_30), ctrl_30.in_peak_window, window_start=0
)
print(
    "30-min event-window timesteps (day 1, expect ~28-32):",
    [t for t in np.where(event_mask_30)[0] if t < 48],
)
print(
    "30-min event-window timesteps (day 2, expect ~72-76):",
    [t for t in np.where(event_mask_30)[0] if t >= 48],
)